In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
import boto3
from botocore.config import Config
from dotenv import load_dotenv
from io import StringIO
import os

In [ ]:
def clean(s):
    return (s or "").strip().replace("\r", "").replace("\n", "")


# Charger variables d'environnement si besoin
load_dotenv()

session = boto3.Session(
    aws_access_key_id=clean(os.getenv("AWS_ACCESS_KEY_ID")),
    aws_secret_access_key=clean(os.getenv("AWS_SECRET_ACCESS_KEY")),
    region_name="eu-west-3",
)
s3 = session.client("s3", config=Config(signature_version="s3v4"))
BUCKET = "mygeodechetbuckets3"


def read_csv_robust(
    body_bytes: bytes, default_sep: str = ",", try_utf8sig_first: bool = True
) -> pd.DataFrame:
    """Lecture robuste : essaie utf-8-sig puis latin-1, set sep, et répare la mojibake si besoin."""
    df = None
    if try_utf8sig_first:
        try:
            df = pd.read_csv(StringIO(body_bytes.decode("utf-8-sig")), sep=default_sep)
        except Exception:
            pass
    if df is None:
        try:
            df = pd.read_csv(StringIO(body_bytes.decode("latin-1")), sep=default_sep)
        except Exception:
            # dernier fallback: utf-8 simple
            df = pd.read_csv(StringIO(body_bytes.decode("utf-8")), sep=default_sep)
    # Nettoyage BOM dans colonnes
    cols = [c.replace("\ufeff", "") for c in df.columns]
    # Répare si mojibake du style DÃ©partement, annÃ©e, ï»¿Code, etc.
    if any(("Ã" in c) or ("ï»¿" in c) or ("Â" in c) for c in cols):
        cols = [
            c.encode("latin-1", "ignore")
            .decode("utf-8", "ignore")
            .replace("\ufeff", "")
            for c in cols
        ]
    df.columns = cols
    return df


# data_wip_v5.csv : UTF-8-SIG, séparateur point-virgule
obj = s3.get_object(Bucket=BUCKET, Key="data_wip_v5.csv")
df = read_csv_robust(obj["Body"].read(), default_sep=";", try_utf8sig_first=True)

In [ ]:
# ===========================================================
# Corrélations PAR RAPPORT AUX 5 CIBLES (déchets)
# Tableau récap Top N + graphiques
# ===========================================================
import pandas as pd

# --- Cibles (les 5 types du CSV) ---
TARGETS_DEFAULT = [
    "Déblais_gravats",
    "Déchets_verts",
    "Encombrants",
    "Matériaux_recyclables",
    "Total_autres_dechets",
]


# --- Helper : gérer les nombres FR (virgules/espaces/NBSP) ---
def _numify(s: pd.Series) -> pd.Series:
    if hasattr(s, "dtype") and s.dtype.kind in "biufc":
        return s
    s = s.astype(str)
    s = (
        s.str.replace("\u00a0", "", regex=False)
        .str.replace(" ", "", regex=False)
        .str.replace(",", ".", regex=False)
    )
    s = s.str.replace(r"[^0-9\.\-eE+]", "", regex=True)
    return pd.to_numeric(s, errors="coerce")


def _coerce_all_numeric_like(d: pd.DataFrame) -> pd.DataFrame:
    d = d.copy()
    for c in d.columns:
        if d[c].dtype.kind in "OUS":
            d[c] = _numify(d[c])
    return d


# -----------------------------------------------------------
# FONCTION PRINCIPALE
# -----------------------------------------------------------
def make_corr_tables_by_targets(
    df: pd.DataFrame,
    targets: list = None,
    *,
    top_k: int = 20,
    rank_by_abs: bool = True,
    exclude_waste_family: bool = True,
    exclude_tonnage_total: bool = False,
    min_non_na: int = 3,
):
    """
    Calcule les corrélations de CHAQUE cible (5 types) avec toutes les autres colonnes numériques.
    Retourne :
      - corr_long : lignes = (Feature, Target, corr, abs_corr, rank)
      - corr_pivot : tableau croisé Features × Targets (valeur = corr signée)
      - corr_topN : pivot trié par moyenne de |corr|, restreint aux Top N (selon moy_abs)
      - corr_top_per_target : dict {target -> DataFrame Top K pour cette target}
    Paramètres clés :
      - top_k : nb de variables retenues par cible (avant union)
      - rank_by_abs : classement par |corr| (True, recommandé) ou corr signée (False)
      - exclude_waste_family : si True, on enlève toutes les colonnes des 5 types (et leurs _n-2)
                               des "features" candidates (pour éviter l’auto-corrélation)
      - exclude_tonnage_total : si True, enlève 'tonnage_dechet_produit' (_n-2) des features
    """
    # 1) cibles présentes
    tgs = [t for t in (targets or TARGETS_DEFAULT) if t in df.columns]
    if not tgs:
        raise ValueError("Aucune des colonnes cibles n'est présente dans le DataFrame.")

    # 2) forcer numérique & corr globale
    dnum = _coerce_all_numeric_like(df)
    # enlever les colonnes quasi vides/constantes
    usable_cols = [
        c
        for c in dnum.columns
        if dnum[c].dtype.kind in "biufc"
        and dnum[c].notna().sum() >= min_non_na
        and dnum[c].nunique(dropna=True) > 1
    ]
    corr = dnum[usable_cols].corr(numeric_only=True)

    # 3) construire la liste des colonnes à exclure des features
    drops = set()
    if exclude_waste_family:
        for base in TARGETS_DEFAULT:
            if base in corr.columns:
                drops.add(base)
            n2 = f"{base}_n-2"
            if n2 in corr.columns:
                drops.add(n2)
    if exclude_tonnage_total:
        for base in ["tonnage_dechet_produit", "tonnage_dechet_produit_n-2"]:
            if base in corr.columns:
                drops.add(base)

    # 4) top K par cible (et union des features retenues)
    long_rows = []
    tops_per_target = {}
    kept_features = set()

    for target in tgs:
        if target not in corr.columns:
            continue
        s = corr[target].drop(labels=[target], errors="ignore")
        if drops:
            s = s.drop(labels=[c for c in drops if c in s.index], errors="ignore")

        # classement
        order = (
            s.abs().sort_values(ascending=False)
            if rank_by_abs
            else s.sort_values(ascending=False)
        )
        top = order.head(top_k).index.tolist()
        kept_features.update(top)

        # table top_k pour cette cible
        df_top = pd.DataFrame(
            {
                "Feature": top,
                "Target": target,
                "corr": [corr.loc[f, target] for f in top],
            }
        )
        df_top["abs_corr"] = df_top["corr"].abs()
        df_top["rank"] = np.arange(1, len(df_top) + 1)
        tops_per_target[target] = df_top

        long_rows.extend(df_top.to_dict("records"))

    # 5) table long globale (union des tops)
    corr_long = pd.DataFrame(long_rows)
    # 6) pivot (features union) × targets (valeurs signées)
    features_sorted = sorted(kept_features)
    pivot = []
    for feat in features_sorted:
        row = {"Feature": feat}
        for t in tgs:
            if (feat in corr.index) and (t in corr.columns):
                row[t] = corr.loc[feat, t]
            else:
                row[t] = np.nan
        pivot.append(row)
    corr_pivot = pd.DataFrame(pivot)

    # 7) moyenne des |corr| et tri global
    if not corr_pivot.empty:
        cols_t = [c for c in tgs if c in corr_pivot.columns]
        corr_pivot["moy_abs"] = corr_pivot[cols_t].abs().mean(axis=1)
        corr_pivot = corr_pivot.sort_values("moy_abs", ascending=False).reset_index(
            drop=True
        )

    # 8) Top N global (par moy_abs)
    corr_topN = corr_pivot.head(top_k).copy()

    return corr_long, corr_pivot, corr_topN, tops_per_target


# -----------------------------------------------------------
# GRAPHIQUES optionnels (barres & heatmap) pour le récap
# -----------------------------------------------------------
def plot_corr_top_features_bars(
    corr_pivot_sorted: pd.DataFrame, targets: list = None, top_n: int = 5, title=None
):
    """Barres groupées: pour les Top N features (par moy_abs), montre corr signée pour chaque target."""
    if corr_pivot_sorted.empty:
        print("Tableau vide.")
        return
    tgs = [t for t in (targets or TARGETS_DEFAULT) if t in corr_pivot_sorted.columns]
    sub = corr_pivot_sorted.head(top_n).copy()
    melted = sub.melt(
        id_vars=["Feature", "moy_abs"] if "moy_abs" in sub.columns else ["Feature"],
        value_vars=tgs,
        var_name="Target",
        value_name="corr",
    ).dropna(subset=["corr"])
    if title is None:
        title = f"Top {top_n} des corrélations des Features sur les Targets"
    fig = px.bar(melted, x="Feature", y="corr", color="Target", title=title)
    fig.update_layout(xaxis_tickangle=-30, yaxis_title="Corrélation")
    fig.show()


def plot_corr_heatmap_pivot(
    corr_pivot_sorted: pd.DataFrame, targets: list = None, top_n: int = 20, title=None
):
    """Heatmap Plotly sur les Top N features (par moy_abs)."""
    if corr_pivot_sorted.empty:
        print("Tableau vide.")
        return
    tgs = [t for t in (targets or TARGETS_DEFAULT) if t in corr_pivot_sorted.columns]
    sub = corr_pivot_sorted.head(top_n).copy()
    if title is None:
        title = f"Table de corrélation — Top {top_n} features vs targets"
    fig = px.imshow(
        sub.set_index("Feature")[tgs], text_auto=True, aspect="auto", title=title
    )
    fig.show()

In [ ]:
# 1) Construire les tableaux (Top 20 par défaut, classement par |corr|)
corr_long, corr_pivot, corr_top20, corr_top_per_target = make_corr_tables_by_targets(
    df,
    top_k=20,
    rank_by_abs=True,  # classe selon |corr|
    exclude_waste_family=True,  # on exclut les 5 colonnes “cibles” des features
    exclude_tonnage_total=False,  # mets True si tu ne veux PAS voir 'tonnage_dechet_produit'
)

In [ ]:
# 2) Afficher les tables
display(corr_long.head(15))  # Feature/Target/corr/abs_corr/rank (long)
display(corr_pivot.head(10))  # pivot complet (trié par moy_abs)
display(corr_top20)

In [ ]:
def make_corr_tables(
    df: pd.DataFrame,
    *,
    targets: list = None,
    top_k: int = 20,
    exclude_waste_family: bool = True,
    exclude_tonnage_total: bool = True,
    exclude_suffixes: tuple = ("_n-2",),
    method: str = "pearson",
    min_non_na: int = 3,
):
    """
    Retourne:
      - corr_pivot: Features × Targets (valeur = corr signée)
      - corr_pivot_sorted: même tableau trié par moyenne(|corr|)
      - corr_long: long (Feature, Target, corr, abs_corr, rank_in_target)
    Règles d'exclusion:
      - exclude_waste_family: enlève les 5 cibles (et leurs *_n-2) des features candidates
      - exclude_tonnage_total: enlève tonnage_dechet_produit (+ *_n-2)
      - exclude_suffixes: enlève toutes les colonnes finissant par ces suffixes (ex: "_n-2")
    """
    tgs = [t for t in (targets or TARGETS_DEFAULT) if t in df.columns]
    if not tgs:
        raise ValueError("Aucune des cibles n'est présente dans le DataFrame.")

    dnum = _coerce_all_numeric_like(df)
    # filtre colonnes numériques suffisantes
    usable = [
        c
        for c in dnum.columns
        if dnum[c].dtype.kind in "biufc"
        and dnum[c].notna().sum() >= min_non_na
        and dnum[c].nunique(dropna=True) > 1
    ]
    dnum = dnum[usable].copy()

    # corr complète
    corr = dnum.corr(method=method, numeric_only=True)

    # set de colonnes exclues des features
    drop_feats = set()
    if exclude_waste_family:
        for base in TARGETS_DEFAULT:
            if base in corr.columns:
                drop_feats.add(base)
            n2 = f"{base}_n-2"
            if n2 in corr.columns:
                drop_feats.add(n2)
    if exclude_tonnage_total:
        for base in ["tonnage_dechet_produit", "tonnage_dechet_produit_n-2"]:
            if base in corr.columns:
                drop_feats.add(base)
    if exclude_suffixes:
        for c in list(corr.columns):
            if any(c.endswith(suf) for suf in exclude_suffixes):
                drop_feats.add(c)

    # TopK par cible + union des features retenues
    kept = set()
    long_rows = []
    for t in tgs:
        if t not in corr.columns:
            continue
        s = corr[t].drop(labels=[t], errors="ignore")
        s = s.drop(labels=[c for c in drop_feats if c in s.index], errors="ignore")
        order = s.abs().sort_values(ascending=False)
        top = order.head(top_k).index.tolist()
        kept.update(top)
        df_top = pd.DataFrame(
            {
                "Feature": top,
                "Target": t,
                "corr": [corr.loc[f, t] for f in top],
            }
        )
        df_top["abs_corr"] = df_top["corr"].abs()
        df_top["rank_in_target"] = np.arange(1, len(df_top) + 1)
        long_rows.extend(df_top.to_dict("records"))

    corr_long = pd.DataFrame(long_rows)

    # pivot complet sur union des features gardées
    pivot_rows = []
    for f in sorted(kept):
        row = {"Feature": f}
        for t in tgs:
            row[t] = (
                corr.loc[f, t] if (f in corr.index and t in corr.columns) else np.nan
            )
        pivot_rows.append(row)
    corr_pivot = pd.DataFrame(pivot_rows)

    # tri global par moyenne(|corr|)
    if not corr_pivot.empty:
        corr_pivot["moy_abs"] = corr_pivot[tgs].abs().mean(axis=1)
        corr_pivot_sorted = corr_pivot.sort_values(
            "moy_abs", ascending=False
        ).reset_index(drop=True)
    else:
        corr_pivot_sorted = corr_pivot.copy()

    return corr_pivot, corr_pivot_sorted, corr_long


# -----------------------------------------------------------
# 1) FIGURE — Bar chart TopN (comme ta capture)
# -----------------------------------------------------------
def plot_topN_bars(
    df: pd.DataFrame,
    *,
    top_n_features: int = 5,
    targets: list = None,
    ignore_tonnage_total: bool = True,
    ignore_n_minus_2: bool = True,
    title: str = "Top {n} des corrélations des Features sur les Targets",
    horizontal: bool = False,  # <-- NEW : barres horizontales si True
    barmode: str = "group",  # "group" ou "stack"
    symmetric_axis: bool = True,  # borne l’axe à [-1, 1]
):
    tgs = [t for t in (targets or TARGETS_DEFAULT) if t in df.columns]
    corr_pivot, corr_sorted, _ = make_corr_tables(
        df,
        targets=tgs,
        top_k=max(top_n_features, 20),
        exclude_waste_family=True,
        exclude_tonnage_total=ignore_tonnage_total,
        exclude_suffixes=("_n-2",) if ignore_n_minus_2 else tuple(),
    )
    if corr_sorted.empty:
        print("Tableau de corrélation vide après exclusions.")
        return

    sub = corr_sorted.head(top_n_features).copy()
    melted = sub.melt(
        id_vars=["Feature", "moy_abs"] if "moy_abs" in sub.columns else ["Feature"],
        value_vars=tgs,
        var_name="Target",
        value_name="corr",
    ).dropna(subset=["corr"])

    if horizontal:
        fig = px.bar(
            melted,
            y="Feature",
            x="corr",
            color="Target",
            orientation="h",
            title=title.format(n=top_n_features),
            labels={"corr": "Valeur de corrélation", "Feature": "Caractéristique"},
        )
        # mettre la feature la plus corrélée en haut
        order = sub["Feature"].tolist()[::-1]
        fig.update_layout(yaxis={"categoryorder": "array", "categoryarray": order})
        if symmetric_axis:
            fig.update_xaxes(range=[-1, 1])
    else:
        fig = px.bar(
            melted,
            x="Feature",
            y="corr",
            color="Target",
            title=title.format(n=top_n_features),
            labels={"corr": "Valeur de corrélation", "Feature": "Caractéristique"},
        )
        if symmetric_axis:
            fig.update_yaxes(range=[-1, 1])
        fig.update_layout(xaxis_tickangle=-25)

    fig.update_layout(barmode=barmode)  # "group" (par défaut) ou "stack"
    fig.show()
    return sub, fig


# -----------------------------------------------------------
# 2) TABLES/MATRICES — en ignorant tonnage & *_n-2
# -----------------------------------------------------------
def build_tables_ignoring_tonnage_n2(
    df: pd.DataFrame,
    *,
    top_k: int = 20,
    targets: list = None,
    method: str = "pearson",
):
    """Renvoie (corr_pivot, corr_topK, corr_long) en excluant tonnage_dechet_produit et *_n-2."""
    tgs = [t for t in (targets or TARGETS_DEFAULT) if t in df.columns]
    corr_pivot, corr_sorted, corr_long = make_corr_tables(
        df,
        targets=tgs,
        top_k=top_k,
        exclude_waste_family=True,
        exclude_tonnage_total=True,  # <-- ignore tonnage
        exclude_suffixes=("_n-2",),  # <-- ignore *_n-2
        method=method,
    )
    corr_topK = corr_sorted.head(top_k).copy()
    return corr_pivot, corr_topK, corr_long


# -----------------------------------------------------------
# 3) HEATMAP — TopN features vs cibles (avec exclusions)
# -----------------------------------------------------------
def plot_heatmap_topN(
    df: pd.DataFrame,
    *,
    top_n_features: int = 20,
    targets: list = None,
    ignore_tonnage_total: bool = True,
    ignore_n_minus_2: bool = True,
    title: str = "Table de corrélation — Top {n} features vs cibles (exclusions appliquées)",
):
    tgs = [t for t in (targets or TARGETS_DEFAULT) if t in df.columns]
    corr_pivot, corr_sorted, _ = make_corr_tables(
        df,
        targets=tgs,
        top_k=max(top_n_features, 20),
        exclude_waste_family=True,
        exclude_tonnage_total=ignore_tonnage_total,
        exclude_suffixes=("_n-2",) if ignore_n_minus_2 else tuple(),
    )
    if corr_sorted.empty:
        print("Rien à afficher.")
        return
    sub = corr_sorted.head(top_n_features).copy()
    fig = px.imshow(
        sub.set_index("Feature")[tgs],
        text_auto=True,
        aspect="auto",
        title=title.format(n=top_n_features),
    )
    fig.show()
    return sub, fig

In [ ]:
# 1) Bar chart “comme ta capture”, en ignorant tonnage + *_n-2
top5_table, fig = plot_topN_bars(
    df, top_n_features=5, ignore_tonnage_total=True, ignore_n_minus_2=True
)

# 2) Tables/Matrices récap (Top20) en ignorant tonnage + *_n-2
corr_pivot, corr_top20, corr_long = build_tables_ignoring_tonnage_n2(df, top_k=20)
display(
    corr_top20
)  # => ton tableau récap Top 20 (features × 5 cibles), trié par moy_abs
# display(corr_pivot) # pivot non tronqué (toutes features gardées)
# display(corr_long)  # long: Feature/Target/corr/abs_corr/rank_in_target

# 3) Heatmap des Top 20 (mêmes exclusions)
sub20, fig_hm = plot_heatmap_topN(
    df, top_n_features=20, ignore_tonnage_total=True, ignore_n_minus_2=True
)

In [ ]:
# ===========================================================
# Heatmap & Bar — Top N features pour UNE cible
# ===========================================================
import pandas as pd


def _prepare_corr_series_for_target(
    df: pd.DataFrame,
    target: str,
    *,
    method: str = "pearson",
    min_non_na: int = 3,
    exclude_waste_family: bool = True,
    ignore_tonnage_total: bool = True,
    ignore_n_minus_2: bool = True,
):
    """Calcule la série de corrélations (toutes features -> target), après exclusions."""
    if target not in df.columns:
        raise ValueError(f"Target '{target}' absente du DataFrame.")

    # force numérique + garde colonnes utilisables
    dnum = _coerce_all_numeric_like(df).copy()
    usable = [
        c
        for c in dnum.columns
        if dnum[c].dtype.kind in "biufc"
        and dnum[c].notna().sum() >= min_non_na
        and dnum[c].nunique(dropna=True) > 1
    ]
    dnum = dnum[usable]
    corr = dnum.corr(method=method, numeric_only=True)

    if target not in corr.columns:
        raise ValueError(
            f"Aucune corrélation calculable pour '{target}' (colonnes non numériques ?)"
        )

    s = corr[target].drop(labels=[target], errors="ignore")

    # exclusions
    drops = set()
    if exclude_waste_family:
        for base in [
            "Déblais_gravats",
            "Déchets_verts",
            "Encombrants",
            "Matériaux_recyclables",
            "Total_autres_dechets",
        ]:
            if base != target:
                drops.add(base)
            if ignore_n_minus_2:
                drops.add(f"{base}_n-2")
    if ignore_tonnage_total:
        drops.update({"tonnage_dechet_produit"})
        if ignore_n_minus_2:
            drops.update({"tonnage_dechet_produit_n-2"})
    if ignore_n_minus_2:
        # retire toutes les colonnes *_n-2 restantes
        drops.update({c for c in s.index if c.endswith("_n-2")})
    s = s.drop(labels=[c for c in drops if c in s.index], errors="ignore")
    return s, corr


def plot_heatmap_target_topN(
    df: pd.DataFrame,
    target: str,
    *,
    top_n: int = 20,
    method: str = "pearson",
    exclude_waste_family: bool = True,
    ignore_tonnage_total: bool = True,
    ignore_n_minus_2: bool = True,
    title: str | None = None,
):
    """
    Heatmap 1 cible × TopN features (ordonnées par |corr| décroissante).
    """
    s, corr = _prepare_corr_series_for_target(
        df,
        target,
        method=method,
        exclude_waste_family=exclude_waste_family,
        ignore_tonnage_total=ignore_tonnage_total,
        ignore_n_minus_2=ignore_n_minus_2,
    )
    if s.empty:
        print("Aucune feature candidate après exclusions.")
        return

    top_feats = s.abs().sort_values(ascending=False).head(top_n).index.tolist()
    mat = corr.loc[top_feats, [target]].copy()
    mat = mat.reindex(mat[target].abs().sort_values(ascending=False).index)

    if title is None:
        title = f"Table de corrélation — Top {top_n} features vs '{target}'"

    fig = px.imshow(
        mat.rename(columns={target: target}), text_auto=True, aspect="auto", title=title
    )
    fig.update_yaxes(title="Feature")
    fig.show()
    return mat, fig


def plot_bars_target_topN(
    df: pd.DataFrame,
    target: str,
    *,
    top_n: int = 20,
    method: str = "pearson",
    exclude_waste_family: bool = True,
    ignore_tonnage_total: bool = True,
    ignore_n_minus_2: bool = True,
    horizontal: bool = True,  # barres renversées par défaut
    symmetric_axis: bool = True,  # borne l’axe à [-1,1]
    title: str | None = None,
):
    """
    Bar chart des TopN features pour une cible (corr signée).
    """
    s, _ = _prepare_corr_series_for_target(
        df,
        target,
        method=method,
        exclude_waste_family=exclude_waste_family,
        ignore_tonnage_total=ignore_tonnage_total,
        ignore_n_minus_2=ignore_n_minus_2,
    )
    if s.empty:
        print("Aucune feature candidate après exclusions.")
        return

    top = s.abs().sort_values(ascending=False).head(top_n)
    tab = (
        pd.DataFrame({"Feature": top.index, "corr": [s.loc[f] for f in top.index]})
        .assign(abs_corr=lambda x: x["corr"].abs())
        .sort_values("abs_corr", ascending=False)
    )
    if title is None:
        title = f"Top {top_n} corrélations vs '{target}'"

    if horizontal:
        fig = px.bar(
            tab,
            y="Feature",
            x="corr",
            orientation="h",
            title=title,
            labels={"corr": "Corrélation"},
        )
        # garder le plus corrélé en haut
        fig.update_layout(
            yaxis={
                "categoryorder": "array",
                "categoryarray": tab["Feature"].tolist()[::-1],
            }
        )
        if symmetric_axis:
            fig.update_xaxes(range=[-1, 1])
    else:
        fig = px.bar(
            tab, x="Feature", y="corr", title=title, labels={"corr": "Corrélation"}
        )
        if symmetric_axis:
            fig.update_yaxes(range=[-1, 1])
        fig.update_layout(xaxis_tickangle=-25)

    fig.show()
    return tab, fig

In [ ]:
# Heatmap Top 20 pour une cible (en ignorant tonnage + *_n-2)
mat, fig = plot_heatmap_target_topN(
    df,
    target="Matériaux_recyclables",
    top_n=20,
    ignore_tonnage_total=True,
    ignore_n_minus_2=True,
)

# Barres horizontales Top 15 pour une cible (Spearman, par ex.)
tab, fig = plot_bars_target_topN(
    df,
    target="Déblais_gravats",
    top_n=15,
    method="spearman",
    ignore_tonnage_total=True,
    ignore_n_minus_2=True,
)